In [22]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv('btc_data_clean.csv', parse_dates=['Open time'])
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
# df.set_index('open_time', inplace=True)
# df.sort_index(inplace=True)

df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2018-01-01 00:00:00,13715.650000,13715.650000,13400.010000,13529.010000,443.356199,2018-01-01 00:59:59.999,5993909.836038,5228,228.521921,3090541.105769,0
1,2018-01-01 01:00:00,13528.990000,13595.890000,13155.380000,13203.060000,383.697006,2018-01-01 01:59:59.999,5154521.555135,4534,180.840403,2430449.492957,0
2,2018-01-01 02:00:00,13203.000000,13418.430000,13200.000000,13330.180000,429.064572,2018-01-01 02:59:59.999,5710192.018530,4887,192.237935,2558504.657645,0
3,2018-01-01 03:00:00,13330.260000,13611.270000,13290.000000,13410.030000,420.087030,2018-01-01 03:59:59.999,5657448.457304,4789,137.918407,1858041.257457,0
4,2018-01-01 04:00:00,13434.980000,13623.290000,13322.150000,13601.010000,340.807329,2018-01-01 04:59:59.999,4588046.996157,4563,172.957635,2328057.920216,0


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71588 entries, 0 to 71587
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   open_time                     71588 non-null  datetime64[ns]
 1   open                          71588 non-null  float64       
 2   high                          71588 non-null  float64       
 3   low                           71588 non-null  float64       
 4   close                         71588 non-null  float64       
 5   volume                        71588 non-null  float64       
 6   close_time                    71588 non-null  object        
 7   quote_asset_volume            71588 non-null  float64       
 8   number_of_trades              71588 non-null  int64         
 9   taker_buy_base_asset_volume   71588 non-null  float64       
 10  taker_buy_quote_asset_volume  71588 non-null  float64       
 11  ignore                      

In [29]:
df['sma_50']  = df['close'].rolling(50).mean()
df['sma_200'] = df['close'].rolling(200).mean()
df['regime']  = (df['sma_50'] > df['sma_200']).astype(int).replace(0, -1)

In [ ]:
# ATR 14
df['tr'] = pd.concat([
    df['high'] - df['low'],
    (df['high'] - df['close'].shift(1)).abs(),
    (df['low']  - df['close'].shift(1)).abs()
], axis=1).max(axis=1)
df['atr_14'] = df['tr'].ewm(span=14, adjust=False).mean()

In [31]:
# NATR (Model feature)
df['natr'] = (df['atr_14'] / df['close']) * 100

In [32]:
# Cleanup & export
df.drop(columns=['tr'], inplace=True)
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore,sma_50,sma_200,regime,atr_14,natr
0,2018-01-01 00:00:00,13715.650000,13715.650000,13400.010000,13529.010000,443.356199,2018-01-01 00:59:59.999,5993909.836038,5228,228.521921,3090541.105769,0,NaN,NaN,-1,315.640000,2.333061
1,2018-01-01 01:00:00,13528.990000,13595.890000,13155.380000,13203.060000,383.697006,2018-01-01 01:59:59.999,5154521.555135,4534,180.840403,2430449.492957,0,NaN,NaN,-1,332.289333,2.516760
2,2018-01-01 02:00:00,13203.000000,13418.430000,13200.000000,13330.180000,429.064572,2018-01-01 02:59:59.999,5710192.018530,4887,192.237935,2558504.657645,0,NaN,NaN,-1,317.108089,2.378873
3,2018-01-01 03:00:00,13330.260000,13611.270000,13290.000000,13410.030000,420.087030,2018-01-01 03:59:59.999,5657448.457304,4789,137.918407,1858041.257457,0,NaN,NaN,-1,317.663010,2.368846
4,2018-01-01 04:00:00,13434.980000,13623.290000,13322.150000,13601.010000,340.807329,2018-01-01 04:59:59.999,4588046.996157,4563,172.957635,2328057.920216,0,NaN,NaN,-1,315.459942,2.319386


In [34]:
feature_cols = ['sma_50', 'sma_200', 'regime', 'atr_14', 'natr']

print(f"new columns: {feature_cols}\n")
print(df[feature_cols].describe().round(4))

new columns: ['sma_50', 'sma_200', 'regime', 'atr_14', 'natr']

             sma_50       sma_200       regime       atr_14         natr
count  71539.000000  71389.000000 71588.000000 71588.000000 71588.000000
mean   38203.642200  38195.064400     0.031200   314.846400     0.927600
std    32486.881000  32469.922400     0.999500   292.357600     0.643300
min     3227.730600   3351.564000    -1.000000     9.556200     0.062400
25%     9545.437300   9484.467600    -1.000000    82.605200     0.538900
50%    28026.328000  28123.266300     1.000000   216.359500     0.777500
75%    59224.177600  59308.269000     1.000000   478.530600     1.110100
max   124089.995800 122296.242000     1.000000  3053.390000    12.851500


In [38]:
print(f"\nRegime split:\n{df['regime'].value_counts().sort_index()}")
print(f"\nNull counts:\n{df[feature_cols].isnull().sum()}")


Regime split:
regime
-1    34677
 1    36911
Name: count, dtype: int64

Null counts:
sma_50      49
sma_200    199
regime       0
atr_14       0
natr         0
dtype: int64


In [ ]:
# export
# df.to_csv('btc_features_sma_atr.csv')
# print(f"Done — {df.shape[0]:,} rows, saved to btc_features_sma_atr.csv")